# Continue School Management Work

This notebook repairs the corrupted `public/app.js` helper and extends the frontend with admin user and parent/guardian support.

In [ ]:
from pathlib import Path
project_root = Path(r'c:\Users\edmun\Desktop\school-management')

app_js = r'''const state = {
  students: [],
  teachers: [],
  classes: [],
  enrollments: [],
  attendance: [],
  grades: [],
  users: [],
  parentLinks: [],
  auth: { token: null, username: null, role: null }
};

function setMessage(text, isError = true) {
  const messageEl = document.getElementById('message');
  messageEl.textContent = text;
  messageEl.style.color = isError ? '#dc2626' : '#064e3b';
}

async function authFetch(path, options = {}) {
  options.headers = options.headers || {};
  if (state.auth.token) {
    options.headers.Authorization = `Bearer ${state.auth.token}`;
  }
  if (options.body && typeof options.body === 'string' && !options.headers['Content-Type']) {
    options.headers['Content-Type'] = 'application/json';
  }

  const res = await fetch(path, options);
  const text = await res.text();
  let body = null;

  try {
    body = JSON.parse(text);
  } catch (_) {
    body = text;
  }

  if (!res.ok) {
    throw new Error(body?.error || body?.message || text || 'Request failed');
  }

  return body;
}

async function loadData() {
  state.students = await authFetch('/api/students');
  state.teachers = await authFetch('/api/teachers');
  state.classes = await authFetch('/api/classes');
  state.enrollments = await authFetch('/api/enrollments');
  state.attendance = await authFetch('/api/attendance');
  state.grades = await authFetch('/api/grades');

  if (state.auth.role === 'admin') {
    state.users = await authFetch('/api/users');
    state.parentLinks = await authFetch('/api/parent-students');
  } else {
    state.users = [];
    state.parentLinks = [];
  }

  renderLists();
}

function buildTable(items, columns) {
  if (!items.length) {
    return '<p>No records found.</p>';
  }

  const headers = columns.map((col) => `<th>${col.label}</th>`).join('');
  const rows = items
    .map((item) =>
      `<tr>${columns
        .map((col) => `<td>${item[col.key] ?? ''}</td>`)
        .join('')}</tr>`
    )
    .join('');

  return `<table><thead><tr>${headers}</tr></thead><tbody>${rows}</tbody></table>`;
}

function renderLists() {
  document.getElementById('students-list').innerHTML = buildTable(state.students, [
    { label: 'ID', key: 'id' },
    { label: 'First Name', key: 'first_name' },
    { label: 'Last Name', key: 'last_name' },
    { label: 'Grade', key: 'grade' }
  ]);

  document.getElementById('teachers-list').innerHTML = buildTable(state.teachers, [
    { label: 'ID', key: 'id' },
    { label: 'First Name', key: 'first_name' },
    { label: 'Last Name', key: 'last_name' },
    { label: 'Subject', key: 'subject' }
  ]);

  document.getElementById('classes-list').innerHTML = buildTable(state.classes, [
    { label: 'ID', key: 'id' },
    { label: 'Class Name', key: 'name' },
    { label: 'Teacher', key: 'teacher_name' },
    { label: 'Room', key: 'room' }
  ]);

  document.getElementById('enrollments-list').innerHTML = buildTable(state.enrollments, [
    { label: 'ID', key: 'id' },
    { label: 'Student', key: 'student_name' },
    { label: 'Class', key: 'class_name' }
  ]);

  document.getElementById('attendance-list').innerHTML = buildTable(state.attendance, [
    { label: 'ID', key: 'id' },
    { label: 'Date', key: 'date' },
    { label: 'Status', key: 'status' },
    { label: 'Student', key: 'student_name' },
    { label: 'Class', key: 'class_name' },
    { label: 'Notes', key: 'notes' }
  ]);

  document.getElementById('grades-list').innerHTML = buildTable(state.grades, [
    { label: 'ID', key: 'id' },
    { label: 'Assignment', key: 'assignment' },
    { label: 'Score', key: 'score' },
    { label: 'Max Score', key: 'max_score' },
    { label: 'Student', key: 'student_name' },
    { label: 'Class', key: 'class_name' },
    { label: 'Date', key: 'date' }
  ]);

  document.getElementById('users-list').innerHTML = buildTable(state.users, [
    { label: 'ID', key: 'id' },
    { label: 'Username', key: 'username' },
    { label: 'Role', key: 'role' },
    { label: 'Teacher ID', key: 'teacher_id' },
    { label: 'Student ID', key: 'student_id' }
  ]);

  document.getElementById('parent-links-list').innerHTML = buildTable(state.parentLinks, [
    { label: 'ID', key: 'id' },
    { label: 'Parent Username', key: 'parent_username' },
    { label: 'Student', key: 'student_name' }
  ]);

  const studentSelects = [
    document.getElementById('student-select'),
    document.getElementById('attendance-student-select'),
    document.getElementById('grade-student-select'),
    document.getElementById('user-student-select'),
    document.getElementById('parent-student-select')
  ];
  const classSelects = [
    document.getElementById('class-select'),
    document.getElementById('attendance-class-select'),
    document.getElementById('grade-class-select')
  ];
  const teacherSelect = document.getElementById('teacher-select');
  const userTeacherSelect = document.getElementById('user-teacher-select');
  const parentUserSelect = document.getElementById('parent-user-select');

  const studentOptions = '<option value="">Select student</option>' +
    state.students.map((student) =>
      `<option value="${student.id}">${student.first_name} ${student.last_name}</option>`
    ).join('');

  const classOptions = '<option value="">Select class</option>' +
    state.classes.map((cls) =>
      `<option value="${cls.id}">${cls.name}</option>`
    ).join('');

  const teacherOptions = '<option value="">Assign teacher</option>' +
    state.teachers.map((teacher) =>
      `<option value="${teacher.id}">${teacher.first_name} ${teacher.last_name}</option>`
    ).join('');

  const parentUserOptions = '<option value="">Select parent account</option>' +
    state.users
      .filter((user) => user.role === 'parent')
      .map((user) => `<option value="${user.id}">${user.username}</option>`)
      .join('');

  studentSelects.forEach((select) => {
    if (select) select.innerHTML = studentOptions;
  });

  classSelects.forEach((select) => {
    if (select) select.innerHTML = classOptions;
  });

  if (teacherSelect) teacherSelect.innerHTML = teacherOptions;
  if (userTeacherSelect) userTeacherSelect.innerHTML = teacherOptions;
  if (parentUserSelect) parentUserSelect.innerHTML = parentUserOptions;
}

function updateAuthUI() {
  const loginCard = document.getElementById('login-card');
  const dashboard = document.getElementById('dashboard');
  const welcomeText = document.getElementById('welcome-text');
  const roleText = document.getElementById('role-text');
  const adminForms = document.getElementById('admin-forms');
  const importCard = document.getElementById('import-card');
  const teacherArea = document.getElementById('teacher-area');
  const viewerNote = document.getElementById('viewer-note');

  const isAdmin = state.auth.role === 'admin';
  const isTeacher = state.auth.role === 'teacher';
  const isParent = state.auth.role === 'parent';
  const isStudent = state.auth.role === 'student';
  const isLoggedIn = Boolean(state.auth.token);

  loginCard.classList.toggle('hidden', isLoggedIn);
  dashboard.classList.toggle('hidden', !isLoggedIn);
  adminForms.classList.toggle('hidden', !isAdmin);
  importCard.classList.toggle('hidden', !isAdmin);
  teacherArea.classList.toggle('hidden', !(isAdmin || isTeacher));

  let note = '';
  if (!isLoggedIn) {
    viewerNote.classList.add('hidden');
  } else if (isAdmin) {
    note = 'Admins can manage school data, accounts, and parent links, and can also import/export CSV files.';
    viewerNote.classList.remove('hidden');
  } else if (isTeacher) {
    note = 'Teachers can add attendance and grade records for their own classes.';
    viewerNote.classList.remove('hidden');
  } else if (isParent) {
    note = 'Parents can view linked student attendance and grade records.';
    viewerNote.classList.remove('hidden');
  } else if (isStudent) {
    note = 'Students can view their own attendance and grade records.';
    viewerNote.classList.remove('hidden');
  }

  viewerNote.textContent = note;
  welcomeText.textContent = isLoggedIn ? `Hello, ${state.auth.username}` : 'Welcome back!';
  roleText.textContent = isLoggedIn ? `Role: ${state.auth.role}` : '';
}

function setAuth(auth) {
  state.auth = auth;
  localStorage.setItem('schoolAuth', JSON.stringify(auth));
}

function clearAuth() {
  state.auth = { token: null, username: null, role: null };
  localStorage.removeItem('schoolAuth');
}

function loadStoredAuth() {
  const stored = localStorage.getItem('schoolAuth');
  if (!stored) {
    return;
  }

  try {
    state.auth = JSON.parse(stored);
  } catch (error) {
    clearAuth();
  }
}

async function downloadCsv(type) {
  try {
    const response = await fetch(`/api/export/${type}`, {
      headers: {
        Authorization: `Bearer ${state.auth.token}`
      }
    });

    if (!response.ok) {
      const errorText = await response.text();
      throw new Error(errorText || 'Export failed');
    }

    const csv = await response.text();
    const blob = new Blob([csv], { type: 'text/csv' });
    const url = URL.createObjectURL(blob);
    const link = document.createElement('a');
    link.href = url;
    link.download = `${type}.csv`;
    document.body.appendChild(link);
    link.click();
    link.remove();
    URL.revokeObjectURL(url);
    setMessage(`Exported ${type} successfully.`, false);
  } catch (error) {
    setMessage(error.message);
  }
}

async function handleImport(event) {
  event.preventDefault();
  const form = event.target;
  const formData = new FormData(form);
  const type = formData.get('type');

  if (!type) {
    setMessage('Select a valid import type.');
    return;
  }

  try {
    await authFetch(`/api/import/${type}`, {
      method: 'POST',
      body: formData
    });
    setMessage(`${type} imported successfully.`, false);
    form.reset();
    await loadData();
  } catch (error) {
    setMessage(error.message);
  }
}

function wireForm(formId, apiPath, payloadBuilder) {
  const form = document.getElementById(formId);
  form.addEventListener('submit', async (event) => {
    event.preventDefault();
    const formData = new FormData(form);
    const payload = payloadBuilder(Object.fromEntries(formData.entries()));

    try {
      await authFetch(apiPath, {
        method: 'POST',
        body: JSON.stringify(payload)
      });
      form.reset();
      await loadData();
      setMessage('Saved successfully.', false);
    } catch (error) {
      setMessage(error.message);
    }
  });
}

window.addEventListener('DOMContentLoaded', async () => {
  loadStoredAuth();
  updateAuthUI();

  document.getElementById('login-form').addEventListener('submit', async (event) => {
    event.preventDefault();
    const formData = new FormData(event.target);
    const username = formData.get('username');
    const password = formData.get('password');

    try {
      const result = await authFetch('/api/login', {
        method: 'POST',
        body: JSON.stringify({ username, password })
      });
      setAuth(result);
      updateAuthUI();
      await loadData();
      setMessage('Login successful.', false);
    } catch (error) {
      setMessage(error.message);
    }
  });

  document.getElementById('logout-button').addEventListener('click', () => {
    clearAuth();
    updateAuthUI();
    setMessage('Logged out.', false);
  });

  document.querySelectorAll('[data-type]').forEach((button) => {
    button.addEventListener('click', () => downloadCsv(button.dataset.type));
  });

  document.getElementById('import-form').addEventListener('submit', handleImport);

  wireForm('student-form', '/api/students', ({ first_name, last_name, grade }) => ({ first_name, last_name, grade }));
  wireForm('teacher-form', '/api/teachers', ({ first_name, last_name, subject }) => ({ first_name, last_name, subject }));
  wireForm('class-form', '/api/classes', ({ name, room, teacher_id }) => ({ name, room, teacher_id: teacher_id || null }));
  wireForm('enrollment-form', '/api/enrollments', ({ student_id, class_id }) => ({ student_id, class_id }));
  wireForm('attendance-form', '/api/attendance', ({ student_id, class_id, date, status, notes }) => ({ student_id, class_id, date, status, notes }));
  wireForm('grade-form', '/api/grades', ({ student_id, class_id, assignment, score, max_score, comments, date }) => ({ student_id, class_id, assignment, score, max_score, comments, date }));
  wireForm('user-form', '/api/users', ({ username, password, role, teacher_id, student_id }) => ({ username, password, role, teacher_id: teacher_id || null, student_id: student_id || null }));
  wireForm('parent-link-form', '/api/parent-students', ({ parent_user_id, student_id }) => ({ parent_user_id, student_id }));

  if (state.auth.token) {
    try {
      await loadData();
    } catch (error) {
      clearAuth();
      updateAuthUI();
      setMessage('Session expired. Please log in again.');
    }
  }
});
'''

index_html = r'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>School Management</title>
  <link rel="stylesheet" href="app.css" />
</head>
<body>
  <header>
    <h1>School Management</h1>
    <p>Secure school administration with login, role-based access, and CSV import/export.</p>
  </header>
  <main>
    <section class="card" id="login-card">
      <h2>Staff login</h2>
      <p>Use default admin credentials: <strong>admin</strong> / <strong>admin123</strong></p>
      <form id="login-form">
        <input name="username" placeholder="Username" required />
        <input type="password" name="password" placeholder="Password" required />
        <button type="submit">Login</button>
      </form>
      <div id="message" class="message"></div>
    </section>

    <section class="card hidden" id="dashboard">
      <div class="toolbar">
        <div>
          <div id="welcome-text">Welcome back!</div>
          <div id="role-text"></div>
        </div>
        <button id="logout-button" type="button">Logout</button>
      </div>

      <div class="note">
        <p>Admins can manage students, teachers, classes, enrollments, accounts, and parent/guardian links; they can also import/export CSV data.</p>
      </div>

      <section id="admin-forms">
        <div class="card">
          <h2>Students</h2>
          <form id="student-form">
            <input name="first_name" placeholder="First name" required />
            <input name="last_name" placeholder="Last name" required />
            <input name="grade" placeholder="Grade" />
            <button type="submit">Add Student</button>
          </form>
          <div id="students-list"></div>
        </div>

        <div class="card">
          <h2>Teachers</h2>
          <form id="teacher-form">
            <input name="first_name" placeholder="First name" required />
            <input name="last_name" placeholder="Last name" required />
            <input name="subject" placeholder="Subject" />
            <button type="submit">Add Teacher</button>
          </form>
          <div id="teachers-list"></div>
        </div>

        <div class="card">
          <h2>Classes</h2>
          <form id="class-form">
            <input name="name" placeholder="Class name" required />
            <input name="room" placeholder="Room" />
            <select name="teacher_id" id="teacher-select">
              <option value="">Assign teacher</option>
            </select>
            <button type="submit">Add Class</button>
          </form>
          <div id="classes-list"></div>
        </div>

        <div class="card">
          <h2>Enrollments</h2>
          <form id="enrollment-form">
            <select name="student_id" id="student-select"></select>
            <select name="class_id" id="class-select"></select>
            <button type="submit">Enroll Student</button>
          </form>
          <div id="enrollments-list"></div>
        </div>

        <div class="card">
          <h2>Staff accounts</h2>
          <form id="user-form">
            <input name="username" placeholder="Username" required />
            <input type="password" name="password" placeholder="Password" required />
            <select name="role" required>
              <option value="">Select role</option>
              <option value="admin">Admin</option>
              <option value="teacher">Teacher</option>
              <option value="student">Student</option>
              <option value="parent">Parent</option>
            </select>
            <select name="teacher_id" id="user-teacher-select">
              <option value="">Assign teacher account</option>
            </select>
            <select name="student_id" id="user-student-select">
              <option value="">Assign student account</option>
            </select>
            <button type="submit">Create User</button>
          </form>
          <div id="users-list"></div>
        </div>

        <div class="card">
          <h2>Parent / Guardian links</h2>
          <form id="parent-link-form">
            <select name="parent_user_id" id="parent-user-select"></select>
            <select name="student_id" id="parent-student-select"></select>
            <button type="submit">Link Parent to Student</button>
          </form>
          <div id="parent-links-list"></div>
        </div>
      </section>

      <section id="teacher-area" class="card hidden">
        <h2>Teacher tools</h2>
        <form id="attendance-form" class="small-form">
          <select name="student_id" id="attendance-student-select"></select>
          <select name="class_id" id="attendance-class-select"></select>
          <input name="date" type="date" required />
          <select name="status" required>
            <option value="">Status</option>
            <option value="present">Present</option>
            <option value="absent">Absent</option>
            <option value="late">Late</option>
          </select>
          <input name="notes" placeholder="Notes" />
          <button type="submit">Record Attendance</button>
        </form>
        <form id="grade-form" class="small-form">
          <select name="student_id" id="grade-student-select"></select>
          <select name="class_id" id="grade-class-select"></select>
          <input name="assignment" placeholder="Assignment" required />
          <input name="score" type="number" placeholder="Score" />
          <input name="max_score" type="number" placeholder="Max score" />
          <input name="date" type="date" />
          <input name="comments" placeholder="Comments" />
          <button type="submit">Add Grade</button>
        </form>
      </section>

      <section class="card">
        <h2>Attendance</h2>
        <div id="attendance-list"></div>
      </section>

      <section class="card">
        <h2>Grades</h2>
        <div id="grades-list"></div>
      </section>

      <div id="viewer-note" class="note hidden"></div>

      <section class="card">
        <h2>Data export</h2>
        <div class="button-row">
          <button type="button" data-type="students">Export students</button>
          <button type="button" data-type="teachers">Export teachers</button>
          <button type="button" data-type="classes">Export classes</button>
          <button type="button" data-type="enrollments">Export enrollments</button>
          <button type="button" data-type="attendance">Export attendance</button>
          <button type="button" data-type="grades">Export grades</button>
        </div>
      </section>

      <section class="card" id="import-card">
        <h2>CSV import</h2>
        <form id="import-form">
          <select name="type" required>
            <option value="">Select import type</option>
            <option value="students">Students</option>
            <option value="teachers">Teachers</option>
            <option value="classes">Classes</option>
            <option value="enrollments">Enrollments</option>
            <option value="attendance">Attendance</option>
            <option value="grades">Grades</option>
          </select>
          <input type="file" name="file" accept=".csv" required />
          <button type="submit">Upload CSV</button>
        </form>
        <p class="hint">Example rows: students: first_name,last_name,grade; attendance: student_id,class_id,date,status,notes</p>
      </section>
    </section>
  </main>
  <script src="app.js"></script>
</body>
</html>
'''

(project_root / 'public' / 'app.js').write_text(app_js, encoding='utf-8')
(project_root / 'public' / 'index.html').write_text(index_html, encoding='utf-8')
print('Rewrote public/app.js and public/index.html')